---
# 2D Batch Normalization Forward Pass (2 Hours)
---

Author: Jan Robine

In this notebook, you will implement the forward pass of a 2D batch normalization layer in a fully vectorized way.

In [2]:
%load_ext autoreload
%autoreload 2
import tests
import train
import visual

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---

## Theory

---

A 2D batch normalization layer takes an input tensor

$$
x \in \mathbb{R}^{N \times C \times H \times W}
$$

and normalizes each **channel** separately.
Here, $N$ is the batch size, $C$ is the number of channels, and $H$ and $W$ are the height and width of the input feature maps.

The batch mean $\mu \in \mathbb{R}^C$ and variance $\sigma^2 \in \mathbb{R}^C$ are computed **per channel**, and the same mean and variance are used to normalize **all spatial positions** of that channel.

For one channel $c$, the batch mean $\mu_c$ and variance $\sigma_c^2$ are computed as follows:

$$
\mu_c = \frac{1}{NHW} \sum_{n=1}^N \sum_{h=1}^H \sum_{w=1}^W x_{n,c,h,w},
\qquad
\sigma_c^2 = \frac{1}{NHW} \sum_{n=1}^N \sum_{h=1}^H \sum_{w=1}^W \left(x_{n,c,h,w} - \mu_c\right)^2.
$$

Then, the input is normalized using the mean and variance:

$$
\hat{x}_{n,c,h,w}
=
\frac{x_{n,c,h,w} - \mu_c}{\sqrt{\sigma_c^2 + \varepsilon}}.
$$
where $\varepsilon > 0$ is a small constant for numerical stability.

Finally, batch normalization applies one learnable scale $\gamma_c \in \mathbb{R}$ and one learnable shift $\beta_c \in \mathbb{R}$ per channel:

$$
y_{n,c,h,w} = \gamma_c \, \hat{x}_{n,c,h,w} + \beta_c,
$$
The full output tensor $y$ has the same shape as the input $x$.

---
## Training vs. Inference
---

Batch normalization behaves differently during training and during inference.

During training, the layer uses the statistics of the current batch, as described above.

During inference, the layer uses **running estimates** of the mean and variance that were collected during training.  
This makes the behavior independent of the current batch and is important when using a model after training.

The running estimates $\bar{\mu}$ and $\bar{\sigma}^2$ are accumulated during training via exponential moving averages.  
For one channel $c$:

$$\begin{aligned}
\bar{\mu}_c & \leftarrow (1-\alpha)\,\bar{\mu}_c + \alpha\,\mu_c \\
\bar{\sigma}^2_c & \leftarrow (1-\alpha)\,\bar{\sigma}^2_c + \alpha\,\hat{\sigma}^2_c,
\end{aligned}$$
where $\alpha \in [0,1]$ is the *momentum* hyperparameter, $\mu_c$ is the batch mean,
and $\hat{\sigma}^2_c$ is the batch variance with denominator $NHW - 1$ instead of $NHW$:
$$\hat{\sigma}^2_c = \frac{1}{NHW - 1} \sum_{n,h,w} \left(x_{n,c,h,w} - \mu_c\right)^2.$$

The variance can be computed with `torch.var`, where the `correction` argument can be used to specify the denominator, with `correction=0` for $NHW$ and `correction=1` for $NHW - 1$.

---

In PyTorch, neural networks (often instances of `torch.nn.Module`) usually switch between these two modes with

`net = MY_BIG_CNN_WITH_LOTS_OF_LAYERS()`
- `net.train()`
- `net.eval()`

and inside a network module this can be checked with `self.training`.  
In this notebook, we do not implement a full PyTorch module. Instead, we use separate functions for training mode and inference/eval mode.

---
## **Task:**

Implement `init_batchnorm2d` in `batchnorm.py`.

Complete the task by creating the channel-wise tensors needed for batch normalization:

- scale tensor `gamma`, initialized with ones,
- shift tensor `beta`, initialized with zeros,
- `running_mean`, initialized with zeros,
- `running_var`, initialized with ones.

Return all four tensors with shape `(C,)`.

In [3]:
tests.test_init_batchnorm2d()

✅ PASS: init_batchnorm2d() is correct.


---
## **Task:**

Implement `batchnorm2d_forward_train` in `batchnorm.py`.

Complete the task by implementing the fully vectorized forward pass for training mode.

Hints:
- You can use `torch.mean` and `torch.var`, which have a `dim` argument to specify the dimensions to reduce over.
- `torch.var` has a `correction` argument to specify the denominator (0 or 1).
- Use **broadcasting** over the batch and spatial dimensions for normalization and the affine transformation.
- For this notebook, you can assume there always are at least 2 values per channel, so that $NHW > 1$.

In [4]:
tests.test_batchnorm2d_forward_train()

✅ PASS: batchnorm2d_forward_train() is correct.


---
## **Task:**

Implement `batchnorm2d_forward_eval` in `batchnorm.py`.

During inference, batch normalization uses the stored running mean and running variance instead of the batch statistics.

Complete the task by implementing the fully vectorized forward pass for inference mode.

In [5]:
tests.test_batchnorm2d_forward_eval()

✅ PASS: batchnorm2d_forward_eval() is correct.


---
## Short CNN Comparison
---

Now compare two small CNNs on a small subset of the CIFAR-10 dataset.  
These are color images with shape $(3, 32, 32)$ and 10 possible classes.

Look at some example images from the dataset.

In [6]:
visual.cifar10_pictures(n=8).show()

100%|██████████| 170M/170M [43:00<00:00, 66.1kB/s]  


The two networks we compare are:
- `BaselineCNN` uses strided convolutions for downsampling and does not use any normalization.
- `BatchNormCNN` uses the same architecture but with batch normalization layers after each convolutional layer.

We tuned the hyperparameters individually for each model to give them a good chance to perform well.

The training run is intentionally short so that it also works on CPU. If CUDA is available, it is used automatically.

In [7]:
models, histories = train.train_short_comparison(
    epochs=5,
    batch_size=128,
    max_train_samples=4000,
    max_test_samples=1000,
)

visual.show_training_comparison(histories)

Using device: cpu

Training baseline


Train loss: 1.451, Test accuracy: 46.50 %: 100%|██████████| 5/5 [00:25<00:00,  5.06s/it]



Training batchnorm


Train loss: 0.903, Test accuracy: 51.30 %: 100%|██████████| 5/5 [00:31<00:00,  6.23s/it]


Final test accuracy: baseline=46.50 %, batchnorm=51.30 %, delta=+4.80 percentage points
